# 00 — Environment Setup

Install dependencies, mount Google Drive, verify GPU, and download benchmark data.

In [ ]:
import subprocess, sys

def install_deps():
    """Install research dependencies from requirements-research.txt."""
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'unsloth', 'trl>=0.12', 'datasets', 'vllm',
        'xgboost', 'scikit-learn', 'scipy', 'seaborn',
        'sqlglot', 'pytest',
    ])
    print('Dependencies installed.')

install_deps()

In [ ]:
import torch

def check_gpu():
    """Print GPU info and verify CUDA availability."""
    if not torch.cuda.is_available():
        print('WARNING: No GPU detected. Training will be very slow.')
        return
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_mem / 1024**3
        print(f'GPU {i}: {props.name}, {vram_gb:.1f} GB VRAM')
        print(f'  Compute capability: {props.major}.{props.minor}')
        print(f'  bf16 support: {props.major >= 8}')
    print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')

check_gpu()

In [ ]:
from pathlib import Path

def setup_drive(project_name='SLM-PRJ'):
    """Mount Google Drive and return project root path."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        root = Path('/content/drive/MyDrive') / project_name
        print(f'Drive mounted. Project root: {root}')
    except ImportError:
        # Kaggle or local — use /kaggle/working or cwd
        kaggle = Path('/kaggle/working')
        root = kaggle / project_name if kaggle.exists() else Path.cwd()
        print(f'Not Colab. Project root: {root}')
    root.mkdir(parents=True, exist_ok=True)
    return root

PROJECT_ROOT = setup_drive()

In [ ]:
import os, subprocess

REPO_DIR = PROJECT_ROOT / 'SLM-PRJ'
if not REPO_DIR.exists():
    # TODO: Replace with your actual GitHub repo URL
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USER/SLM-PRJ.git', str(REPO_DIR)], check=True)
    print(f'Cloned repo to {REPO_DIR}')
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    print(f'Pulled latest at {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f'Working directory: {os.getcwd()}')

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

BIRD_DIR = DATA_DIR / 'bird'
SPIDER_DIR = DATA_DIR / 'spider'

# BIRD: download from Kaggle dataset or manual upload
# Spider: download from Yale SPIDER dataset
# SynSQL: download from HuggingFace gretelai/synthetic-text-to-sql

print('Data directories:')
print(f'  BIRD:   {BIRD_DIR}')
print(f'  Spider: {SPIDER_DIR}')
print(f'  SynSQL: {DATA_DIR / "synsql"}')
print()
print('TODO: Upload/download datasets to these directories.')
print('  BIRD dev.json + dev_databases/ -> data/bird/')
print('  Spider dev.json + database/    -> data/spider/')
print('  SynSQL raw JSONL               -> data/synsql/raw.jsonl')

In [ ]:
# Verify core modules load correctly
from benchmarks.bird_loader import load_bird_dev, get_db_path
from benchmarks.spider_loader import load_spider_dev
from benchmarks.eval_ex import execution_match, batch_evaluate
from training.data_prep import filter_synsql, prepare_subsets
from training.prompt_template import format_sft_messages

print('All imports OK.')

In [ ]:
# Run unit tests
!python -m pytest tests/test_eval_ex.py tests/test_data_prep.py -v